Bước 1. Đọc data từ HDFS

In [43]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Pakistan_Ecommerce_BigData") \
    .getOrCreate()

In [44]:
hdfs_path = "hdfs://localhost:9000/ecom/Pakistan_Largest_Ecommerce_Dataset.csv"

df = spark.read.csv(
    hdfs_path,
    header=True,
    inferSchema=True
)

df.show(5)

+-------+--------------+----------+--------------------+------+-----------+-----------+------------+-----------------+---------------------+---------------+--------------+------------+---------+-------+----+-----+--------------+------+----+-----------+----+----+----+----+----+
|item_id|        status|created_at|                 sku| price|qty_ordered|grand_total|increment_id|  category_name_1|sales_commission_code|discount_amount|payment_method|Working Date|BI Status|    MV |Year|Month|Customer Since|   M-Y|  FY|Customer ID|_c21|_c22|_c23|_c24|_c25|
+-------+--------------+----------+--------------------+------+-----------+-----------+------------+-----------------+---------------------+---------------+--------------+------------+---------+-------+----+-----+--------------+------+----+-----------+----+----+----+----+----+
| 211131|      complete|  7/1/2016|   kreations_YI 06-L|1950.0|          1|       1950|   100147443|  Women's Fashion|                   \N|              0|          

Bước 2. Kiểm tra số dòng, số cột ban đầu

In [45]:
print("Số dòng ban đầu:", df.count())
print("Số cột ban đầu:", len(df.columns))
print(df.columns)

Số dòng ban đầu: 1048586
Số cột ban đầu: 26
['item_id', 'status', 'created_at', 'sku', 'price', 'qty_ordered', 'grand_total', 'increment_id', 'category_name_1', 'sales_commission_code', 'discount_amount', 'payment_method', 'Working Date', 'BI Status', ' MV ', 'Year', 'Month', 'Customer Since', 'M-Y', 'FY', 'Customer ID', '_c21', '_c22', '_c23', '_c24', '_c25']


Bước 3. Đổi tên cột

Dataset này có nhiều cột có khoảng trắng và viết hoa như Customer ID, Working Date, BI Status, nên nên đổi về dạng dễ dùng.

In [46]:
import re

def clean_column_name(name):
    name = name.strip()
    name = name.lower()
    name = re.sub(r"[^a-zA-Z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name)
    name = name.strip("_")
    return name

new_columns = [clean_column_name(c) for c in df.columns]
df_clean = df.toDF(*new_columns)
df_clean.show(5)

+-------+--------------+----------+--------------------+------+-----------+-----------+------------+-----------------+---------------------+---------------+--------------+------------+---------+-------+----+-----+--------------+------+----+-----------+----+----+----+----+----+
|item_id|        status|created_at|                 sku| price|qty_ordered|grand_total|increment_id|  category_name_1|sales_commission_code|discount_amount|payment_method|working_date|bi_status|     mv|year|month|customer_since|   m_y|  fy|customer_id| c21| c22| c23| c24| c25|
+-------+--------------+----------+--------------------+------+-----------+-----------+------------+-----------------+---------------------+---------------+--------------+------------+---------+-------+----+-----+--------------+------+----+-----------+----+----+----+----+----+
| 211131|      complete|  7/1/2016|   kreations_YI 06-L|1950.0|          1|       1950|   100147443|  Women's Fashion|                   \N|              0|          

Bước 4. Xóa các cột rác

In [47]:
cols_to_drop = ["c21", "c22", "c23", "c24", "c25"]

cols_to_drop = [c for c in cols_to_drop if c in df_clean.columns]

df_clean = df_clean.drop(*cols_to_drop)

print("Các cột đã xóa:", cols_to_drop)
print("Số cột sau khi xóa:", len(df_clean.columns))

Các cột đã xóa: ['c21', 'c22', 'c23', 'c24', 'c25']
Số cột sau khi xóa: 21


Bước 5. Xóa các dòng trắng ở cuối file

In [48]:
core_cols = [
    "item_id",
    "created_at",
    "sku",
    "price",
    "qty_ordered",
    "grand_total"
]

core_cols = [c for c in core_cols if c in df_clean.columns]

before = df_clean.count()

df_clean = df_clean.dropna(how="all", subset=core_cols)

after = df_clean.count()

print("Số dòng trước khi xóa dòng trắng:", before)
print("Số dòng sau khi xóa dòng trắng:", after)
print("Số dòng trắng đã xóa:", before - after)

Số dòng trước khi xóa dòng trắng: 1048586
Số dòng sau khi xóa dòng trắng: 584535
Số dòng trắng đã xóa: 464051


Bước 6. Kiểm tra giá trị thiếu

In [49]:
from pyspark.sql.functions import col, sum as spark_sum, when

missing_df = df_clean.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_clean.columns
])

missing_df.show(truncate=False)

+-------+------+----------+---+-----+-----------+-----------+------------+---------------+---------------------+---------------+--------------+------------+---------+---+----+-----+--------------+---+---+-----------+
|item_id|status|created_at|sku|price|qty_ordered|grand_total|increment_id|category_name_1|sales_commission_code|discount_amount|payment_method|working_date|bi_status|mv |year|month|customer_since|m_y|fy |customer_id|
+-------+------+----------+---+-----+-----------+-----------+------------+---------------+---------------------+---------------+--------------+------------+---------+---+----+-----+--------------+---+---+-----------+
|0      |15    |0         |20 |11   |11         |11         |11          |175            |137182               |11             |11            |11          |11       |11 |11  |11   |11            |22 |22 |22         |
+-------+------+----------+---+-----+-----------+-----------+------------+---------------+---------------------+---------------+----

Bước 7. Xử lý tiếp các dòng còn thiếu ở cột quan trọng
Bước 

In [50]:
important_cols = [
    "item_id",
    "status",
    "created_at",
    "sku",
    "price",
    "qty_ordered",
    "grand_total",
    "category_name_1",
    "payment_method"
]

important_cols = [c for c in important_cols if c in df_clean.columns]

before = df_clean.count()

df_clean = df_clean.dropna(subset=important_cols)

after = df_clean.count()

print("Số dòng trước:", before)
print("Số dòng sau:", after)
print("Số dòng bị xóa:", before - after)

Số dòng trước: 584535
Số dòng sau: 584325
Số dòng bị xóa: 210


Bước 8. Kiểm tra lại số dòng cuối

In [51]:
print("Số dòng sau làm sạch:", df_clean.count())
print("Số cột sau làm sạch:", len(df_clean.columns))

df_clean.show(5)

Số dòng sau làm sạch: 584325
Số cột sau làm sạch: 21
+-------+--------------+----------+--------------------+------+-----------+-----------+------------+-----------------+---------------------+---------------+--------------+------------+---------+-------+----+-----+--------------+------+----+-----------+
|item_id|        status|created_at|                 sku| price|qty_ordered|grand_total|increment_id|  category_name_1|sales_commission_code|discount_amount|payment_method|working_date|bi_status|     mv|year|month|customer_since|   m_y|  fy|customer_id|
+-------+--------------+----------+--------------------+------+-----------+-----------+------------+-----------------+---------------------+---------------+--------------+------------+---------+-------+----+-----+--------------+------+----+-----------+
| 211131|      complete|  7/1/2016|   kreations_YI 06-L|1950.0|          1|       1950|   100147443|  Women's Fashion|                   \N|              0|           cod|    7/1/2016|    

Bước 9. Làm sạch ngày tháng

In [52]:
from pyspark.sql.functions import col, trim, regexp_extract, concat_ws, when, lpad, to_date

df_clean = df_clean.withColumn(
    "created_at_raw",
    trim(col("created_at").cast("string"))
)

# Dữ liệu đang dạng 7/1/2016 hoặc 12/31/2016
pattern = r"^(\d{1,2})/(\d{1,2})/(\d{4})$"

month_str = regexp_extract(col("created_at_raw"), pattern, 1)
day_str = regexp_extract(col("created_at_raw"), pattern, 2)
year_str = regexp_extract(col("created_at_raw"), pattern, 3)

df_clean = df_clean.withColumn(
    "order_year",
    when(year_str != "", year_str.cast("int")).otherwise(None)
)

df_clean = df_clean.withColumn(
    "order_month",
    when(month_str != "", month_str.cast("int")).otherwise(None)
)

df_clean = df_clean.withColumn(
    "order_day",
    when(day_str != "", day_str.cast("int")).otherwise(None)
)

df_clean = df_clean.withColumn(
    "created_date",
    when(
        year_str != "",
        to_date(
            concat_ws(
                "-",
                year_str,
                lpad(month_str, 2, "0"),
                lpad(day_str, 2, "0")
            ),
            "yyyy-MM-dd"
        )
    ).otherwise(None)
)

df_clean.select(
    "created_at_raw",
    "created_date",
    "order_year",
    "order_month",
    "order_day"
).show(20, truncate=False)

+--------------+------------+----------+-----------+---------+
|created_at_raw|created_date|order_year|order_month|order_day|
+--------------+------------+----------+-----------+---------+
|7/1/2016      |2016-07-01  |2016      |7          |1        |
|7/1/2016      |2016-07-01  |2016      |7          |1        |
|7/1/2016      |2016-07-01  |2016      |7          |1        |
|7/1/2016      |2016-07-01  |2016      |7          |1        |
|7/1/2016      |2016-07-01  |2016      |7          |1        |
|7/1/2016      |2016-07-01  |2016      |7          |1        |
|7/1/2016      |2016-07-01  |2016      |7          |1        |
|7/1/2016      |2016-07-01  |2016      |7          |1        |
|7/1/2016      |2016-07-01  |2016      |7          |1        |
|7/1/2016      |2016-07-01  |2016      |7          |1        |
|7/1/2016      |2016-07-01  |2016      |7          |1        |
|7/1/2016      |2016-07-01  |2016      |7          |1        |
|7/1/2016      |2016-07-01  |2016      |7          |1  

Bước 10. Loại bỏ dòng giá trị không hợp lệ

In [53]:
from pyspark.sql.functions import expr

if "price" in df_clean.columns:
    df_clean = df_clean.filter(expr("try_cast(`price` as double) >= 0"))

if "qty_ordered" in df_clean.columns:
    df_clean = df_clean.filter(expr("try_cast(`qty_ordered` as double) > 0"))

if "grand_total" in df_clean.columns:
    df_clean = df_clean.filter(expr("try_cast(`grand_total` as double) >= 0"))

print("Số dòng sau khi lọc giá trị không hợp lệ:", df_clean.count())

Số dòng sau khi lọc giá trị không hợp lệ: 584238


Bước 11. Chuẩn hóa chữ trong các cột phân loại

In [54]:
from pyspark.sql.functions import lower, trim

text_cols = ["status", "category_name_1", "payment_method", "bi_status"]

for c in text_cols:
    if c in df_clean.columns:
        df_clean = df_clean.withColumn(c, lower(trim(col(c))))

df_clean.select([c for c in text_cols if c in df_clean.columns]).show(10)

+--------------+-----------------+--------------+---------+
|        status|  category_name_1|payment_method|bi_status|
+--------------+-----------------+--------------+---------+
|      complete|  women's fashion|           cod|    #ref!|
|      canceled|beauty & grooming|           cod|    gross|
|      canceled|  women's fashion|           cod|    gross|
|      complete|beauty & grooming|           cod|      net|
|order_refunded|          soghaat|           cod|    valid|
|      canceled|          soghaat|           cod|    gross|
|      complete|beauty & grooming|           cod|      net|
|      complete|          soghaat|           cod|      net|
|      canceled|mobiles & tablets| ublcreditcard|    gross|
|      canceled|mobiles & tablets|     mygateway|    gross|
+--------------+-----------------+--------------+---------+
only showing top 10 rows


Bước 12. Xóa dòng trùng lặp

In [55]:
before = df_clean.count()

df_clean = df_clean.dropDuplicates()

after = df_clean.count()

print("Số dòng trước khi xóa trùng:", before)
print("Số dòng sau khi xóa trùng:", after)
print("Số dòng bị xóa:", before - after)

Số dòng trước khi xóa trùng: 584238
Số dòng sau khi xóa trùng: 584238
Số dòng bị xóa: 0


Bước 13. Tạo thêm cột phục vụ phân tích

In [59]:
from pyspark.sql.functions import col, to_date, year, month, dayofmonth

if "created_at" in df_clean.columns:
    df_clean = df_clean.withColumn(
        "created_date",
        to_date(col("created_at"), "M/d/yyyy")
    )

    df_clean = df_clean.withColumn("order_year", year(col("created_date")))
    df_clean = df_clean.withColumn("order_month", month(col("created_date")))
    df_clean = df_clean.withColumn("order_day", dayofmonth(col("created_date")))

df_clean.select(
    "created_at",
    "created_date",
    "order_year",
    "order_month",
    "order_day"
).show(5, truncate=False)

+----------+------------+----------+-----------+---------+
|created_at|created_date|order_year|order_month|order_day|
+----------+------------+----------+-----------+---------+
|7/1/2016  |2016-07-01  |2016      |7          |1        |
|7/2/2016  |2016-07-02  |2016      |7          |2        |
|7/2/2016  |2016-07-02  |2016      |7          |2        |
|7/2/2016  |2016-07-02  |2016      |7          |2        |
|7/3/2016  |2016-07-03  |2016      |7          |3        |
+----------+------------+----------+-----------+---------+
only showing top 5 rows


In [60]:
helper_cols = [
    "created_at_raw",
    "working_date_raw",
    "customer_since_raw",
    "created_date"
]

helper_cols = [c for c in helper_cols if c in df_clean.columns]

df_clean = df_clean.drop(*helper_cols)

print("Đã xóa cột phụ:", helper_cols)
print("Số cột sau khi xóa cột phụ:", len(df_clean.columns))

Đã xóa cột phụ: ['created_date']
Số cột sau khi xóa cột phụ: 24


Bước 14. Kiểm tra cột sau khi làm sạch

In [61]:
print("Số dòng sau làm sạch:", df_clean.count())
print("Số cột sau làm sạch:", len(df_clean.columns))

df_clean.show(10)

Số dòng sau làm sạch: 584238
Số cột sau làm sạch: 24
+-------+--------------+----------+--------------------+------+-----------+-----------+------------+-----------------+---------------------+---------------+--------------+------------+---------+-------+----+-----+--------------+------+----+-----------+----------+-----------+---------+
|item_id|        status|created_at|                 sku| price|qty_ordered|grand_total|increment_id|  category_name_1|sales_commission_code|discount_amount|payment_method|working_date|bi_status|     mv|year|month|customer_since|   m_y|  fy|customer_id|order_year|order_month|order_day|
+-------+--------------+----------+--------------------+------+-----------+-----------+------------+-----------------+---------------------+---------------+--------------+------------+---------+-------+----+-----+--------------+------+----+-----------+----------+-----------+---------+
| 211400|        refund|  7/1/2016|    Xenium_TG-201653| 300.0|          1|       1399|  

Bước 15. Tải lại data sạch lên HDFS

In [62]:
output_path = "hdfs://localhost:9000/ecom/ecom_clean_csv"

df_clean.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(output_path)

print("Đã lưu dữ liệu sạch dạng CSV vào:", output_path)

Đã lưu dữ liệu sạch dạng CSV vào: hdfs://localhost:9000/ecom/ecom_clean_csv


In [63]:
hdfs_path = "hdfs://localhost:9000/ecom/ecom_clean_csv"

df = spark.read.csv(
    hdfs_path,
    header=True,
    inferSchema=True
)

df.show(5)

+-------+--------+----------+--------------------+------+-----------+-----------+------------+-----------------+---------------------+---------------+---------------+------------+---------+-----+----+-----+--------------+------+----+-----------+----------+-----------+---------+
|item_id|  status|created_at|                 sku| price|qty_ordered|grand_total|increment_id|  category_name_1|sales_commission_code|discount_amount| payment_method|working_date|bi_status|   mv|year|month|customer_since|   m_y|  fy|customer_id|order_year|order_month|order_day|
+-------+--------+----------+--------------------+------+-----------+-----------+------------+-----------------+---------------------+---------------+---------------+------------+---------+-----+----+-----+--------------+------+----+-----------+----------+-----------+---------+
| 211133|canceled|  7/1/2016|kcc_Buy 2 Frey Ai...| 240.0|          1|      240.0|   100147444|beauty & grooming|                   \N|            0.0|            c